# 1. Import and Hardware Setup

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader, Subset

!pip install tqdm ipywidgets -q
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = './Data'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.2 MB/s eta 0:00:00
cuda
cuda


# 2. Hyperparameter

In [2]:
IMG_SIZE = 224
IN_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLASSES = 101

EPOCHS = 150
SEED = 42
LR = 1e-3

MODEL_NAME = 'small'
WIDTH_MULTI = 1.0
DROPOUT = 0.2

# 3. Data Preparation

In [3]:
stats = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [4]:
import os
import random
import numpy as np

def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

set_seed(SEED)

In [5]:
# Download dummy data without transforms
dummy_data = datasets.Food101(root=DATA_PATH, split="train", download=True)

# Split dummy data into two temporary subsets
train_size = int(0.8 * len(dummy_data))
val_size = len(dummy_data) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_tmp_subset, val_tmp_subset = random_split(
    dummy_data, [train_size, val_size], generator=split_generator
)

# Extract the indices
train_indices = train_tmp_subset.indices
val_indices = val_tmp_subset.indices

# Create the correct subsets with transform
train_data = datasets.Food101(
    root=DATA_PATH, split="train", download=False, transform=train_transform
)
val_data = datasets.Food101(
    root=DATA_PATH, split="train", download=False, transform=test_transform
)

train_subset = Subset(train_data, train_indices)
val_subset = Subset(val_data, val_indices)

# Download test dataset
test_dataset = datasets.Food101(
    root=DATA_PATH, split="test", download=True, transform=test_transform
)

100%|██████████| 5.00G/5.00G [04:04<00:00, 20.4MB/s] 



In [ ]:
def seed_worker(worker_id):
    worker_seed = worker_id + SEED
    random.seed(worker_seed)
    np.random.seed(worker_seed)

train_generator = torch.Generator().manual_seed(SEED)
eval_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    dataset=train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    worker_init_fn=seed_worker,
    generator=train_generator,
)

val_loader = DataLoader(
    dataset=val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=1,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=1,
    worker_init_fn=seed_worker,
    generator=eval_generator,
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


<div style="display:flex; gap:10px; align-items:center;">
    <img src="figures/MobileNetV3_SE.png" alt="MobileNetV3n" style="width:51.5%;"/>
    <img src="figures/MobileNetV3_last_stage.png" alt="MobileNetV3" style="width:48.5%;"/>
</div>

<div style="display:flex; gap:10px; align-items:center;">
    <img src="figures/MobileNetV3_small.png" alt="MobileNetV3" style="width:54%;"/>
    <img src="figures/MobileNetV3_large.png" alt="MobileNetV3" style="width:40%;"/>
</div>

# 4. Model Architecture

In [7]:
def _make_divisible(old_value, divisor=8, min_value=None):
    # Set
    if min_value is None:
        min_value = divisor

    # Calculate new value
    new_value = max(min_value, int(old_value + divisor / 2) // divisor * divisor)

    # Make sure that the new value doesn't drop too far below
    if new_value < 0.9 * old_value:
        new_value += divisor
    return new_value


class Conv2dNormActivation(nn.Sequential):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        groups=1,
        activation=nn.ReLU,
    ):
        padding = (kernel_size - 1) // 2
        activation = (
            nn.Identity() if activation == nn.Identity() else activation(inplace=True)
        )
        super().__init__(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            activation,
        )


# Avg -> FC -> ReLU -> FC -> HardSigmoid
# Use conv2d 1x1 instead of Linear to stay clean, since a 1x1 conv
# is mathematically like a linear layer
class SqueezeExcitation(nn.Sequential):
    def __init__(self, in_channels, squeeze_factor=4):
        super().__init__()
        squeeze_channels = _make_divisible(in_channels // squeeze_factor, divisor=8)
        self.block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=squeeze_channels,
                kernel_size=1,
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels=squeeze_channels,
                out_channels=in_channels,
                kernel_size=1,
            ),
            nn.Hardsigmoid(inplace=True),
        )

    def forward(self, x):
        return x * self.block(x)


class EfficientLastStage(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.2):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.Hardswish(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, num_classes),
        )
    
    def forward(self, x):
        x = self.avg(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class BottleNeck(nn.Module):
    def __init__(
        self,
        in_channels,
        expanded_channels,
        out_channels,
        kernel_size,
        use_se,
        use_hs,
        stride,
    ):
        super().__init__()
        assert stride in [1, 2]
        activation = nn.Hardswish if use_hs else nn.ReLU
        self.use_res_connect = stride == 1 and in_channels == out_channels

        layers = []
        # First 1x1 conv
        layers.append(
            Conv2dNormActivation(
                in_channels=in_channels,
                out_channels=expanded_channels,
                kernel_size=1,
                stride=1,
                groups=1,
                activation=activation,
            )
        )

        # Depthwise conv
        layers.append(
            Conv2dNormActivation(
                in_channels=expanded_channels,
                out_channels=expanded_channels,
                kernel_size=kernel_size,
                stride=stride,
                groups=expanded_channels,
                activation=activation,
            )
        )

        # SqueezeExcitation
        if use_se:
            layers.append(SqueezeExcitation(expanded_channels))

        # 1x1 conv
        layers.append(
            Conv2dNormActivation(
                in_channels=expanded_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,
                activation=nn.Identity,
            )
        )

        # Execute the layers
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        return out + x if self.use_res_connect else out


class MobileNetV3(nn.Module):
    def __init__(
        self, in_channels, num_classes, model_name, width_multi=1.0, dropout=0.2
    ):
        super().__init__()
        # First conv
        layers = [
            Conv2dNormActivation(
                in_channels=in_channels,
                out_channels=16,
                kernel_size=3,
                stride=2,
                activation=nn.Hardswish,
            )
        ]

        # Create configs for small and large version for bottleneck blocks
        model_cfg = {
            "small": {
                "configs": [
                    [3, 16, 16, True, False, 2],
                    [3, 72, 24, False, False, 2],
                    [3, 88, 24, False, False, 1],
                    [5, 96, 40, True, True, 2],
                    [5, 240, 40, True, True, 1],
                    [5, 240, 40, True, True, 1],
                    [5, 120, 48, True, True, 1],
                    [5, 144, 48, True, True, 1],
                    [5, 288, 96, True, True, 2],
                    [5, 576, 96, True, True, 1],
                    [5, 576, 96, True, True, 1],
                ],
                "last_conv_channels": 576,
                "last_cls_channels": 1024,
            },
            "large": {
                "configs": [
                    [3, 16, 16, False, False, 1],
                    [3, 64, 24, False, False, 2],
                    [3, 72, 24, False, False, 2],
                    #
                    [5, 72, 40, True, False, 2],
                    [5, 120, 40, True, False, 1],
                    [5, 120, 40, True, False, 1],
                    #
                    [3, 240, 80, False, True, 2],
                    [3, 200, 80, False, True, 1],
                    [3, 184, 80, False, True, 1],
                    #
                    [3, 184, 80, False, True, 1],
                    [3, 480, 112, True, True, 1],
                    [3, 672, 112, True, True, 1],
                    #
                    [5, 672, 160, True, True, 2],
                    [5, 960, 160, True, True, 1],
                    [5, 960, 160, True, True, 1],
                ],
                "last_conv_channels": 960,
                "last_cls_channels": 1280,
            },
        }

        if model_name not in model_cfg:
            raise ValueError(f"model_name must be 'large' or 'small'. provided: {model_name}")

        # Extract config values
        cfgs = model_cfg[model_name]["configs"]
        last_conv_channels = model_cfg[model_name]["last_conv_channels"]
        last_cls_channels = model_cfg[model_name]["last_cls_channels"]

        # Use lambda function for convinience
        c = lambda ch: _make_divisible(ch * width_multi, divisor=8)
        first_channels = c(16)

        # The bottleneck blocks
        in_channels = first_channels
        for k, exp, out, se, hs, s in cfgs:
            layers.append(
                BottleNeck(
                    in_channels=in_channels,
                    expanded_channels=c(exp),
                    out_channels=c(out),
                    kernel_size=k,
                    use_se=se,
                    use_hs=hs,
                    stride=s,
                )
            )
            in_channels = c(out)
            
        # The last conv 1x1
        layers.append(
            Conv2dNormActivation(
                in_channels=in_channels,
                out_channels=c(last_conv_channels),
                kernel_size=1,
                activation=nn.Hardswish,
            )
        )
        
        self.features = nn.Sequential(*layers)
        
        # Efficient Last Stage
        hidden_channels = _make_divisible(last_cls_channels * max(1.0, width_multi), 8)
        self.head = EfficientLastStage(
            in_channels=c(last_conv_channels),
            hidden_channels=hidden_channels,
            num_classes=num_classes,
            dropout=dropout,
        )
        
        # Weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)
        
    def forward(self, x):
        x = self.features(x)
        return self.head(x)

In [8]:
model = MobileNetV3(
    in_channels=IN_CHANNELS,
    num_classes=NUM_CLASSES,
    model_name=MODEL_NAME,
    width_multi=WIDTH_MULTI,
    dropout=DROPOUT,
).to(device)

print(f"Total parameters: {(sum(p.numel() for p in model.parameters()) / 1e6):.2f}M")

# Use multiple GPUs if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

Total parameters: 1.62M


# 5. Training Preparation

In [9]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0, save_path='best_checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.save_path = save_path
        self.early_stop = False
        self.counter = 0
        self.verbose = False
        self.best_loss = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss >= self.best_loss - self.delta:
            self.counter += 1
            print(f"Early Stopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.counter = 0
            self.best_loss = val_loss
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        if self.verbose:
            print('Saving best checkpoint ...')
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, self.save_path)

In [10]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    div_factor=10,
    final_div_factor=100
)

scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

In [11]:
def train(model, loader, criterion, optimizer, scaler, scheduler):
    model.train()
    loop = tqdm(loader, desc='Training', leave=False)
    train_loss, train_acc = 0, 0
    
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        # Get prediction and loss using Mixed Precision
        with torch.amp.autocast(device_type=device.type):
            out = model(x)
            loss = criterion(out, y)
            
        # Scale up the loss and backpropagate
        scaler.scale(loss).backward()
        
        # Scale down the gradient and clip it
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 
        scaler.step(optimizer)
        
        # Update the scale factor and scheduler (OneCycleLR needs per-batch step)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.detach() * x.size(0)
        train_acc += (out.argmax(1) == y).sum()
        
    return train_loss.item() / len(loader.dataset), train_acc.item() / len(loader.dataset)

def validate(model, loader, criterion):
    model.eval()
    val_loss, val_acc = 0, 0
    loop = tqdm(loader, desc='Validation', leave=False)

    for x, y in loop:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            out = model(x)
            loss = criterion(out, y)
        val_loss += loss.detach() * x.size(0)
        val_acc += (out.argmax(1) == y).sum()

    return val_loss.item() / len(loader.dataset), val_acc.item() / len(loader.dataset)

def test(model, loader):
    model.eval()
    test_acc = 0
    loop = tqdm(loader, desc='Testing', leave=False)

    for x, y in loop:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            out = model(x)
        test_acc += (out.argmax(1) == y).sum()

    return test_acc.item() / len(loader.dataset)

# 6. Train

In [ ]:
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
early_stopping = EarlyStopping(patience=5)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, scaler, scheduler)
    val_loss, val_acc = validate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}: train_loss: {train_loss:.4f}, val_loss: {val_loss:.4f}, " +
          f"train_acc: {train_acc:.4f}, val_acc: {val_acc:.4f}")

    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early Stopping")
        break

best_model = model.module if hasattr(model, "module") else model
best_model.load_state_dict(torch.load("best_checkpoint.pth", map_location=device))
test_acc = test(model, test_loader)
print(f"Final test accuracy: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Training:   0%|          | 0/474 [00:00<?, ?it/s]

# 7. GradCAM

In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.fwd_handle = target_layer.register_forward_hook(self._forward_hook)
        self.bwd_handle = target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inp, out):
        self.activations = out.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1)
        score = logits[torch.arange(logits.size(0), device=logits.device), class_idx].sum()
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)

        cam_min = cam.amin(dim=(2, 3), keepdim=True)
        cam_max = cam.amax(dim=(2, 3), keepdim=True)
        cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
        return cam, logits

    def close(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()

def denormalize(img, mean, std):
    mean = torch.tensor(mean, device=img.device).view(1, 3, 1, 1)
    std = torch.tensor(std, device=img.device).view(1, 3, 1, 1)
    return torch.clamp(img * std + mean, 0, 1)

# Setup GradCAM on last feature block
model_for_cam = model.module if hasattr(model, 'module') else model
gradcam = GradCAM(model_for_cam, model_for_cam.features[-1])

# Sample one image from test set
sample_x, sample_y = test_dataset[0]
x = sample_x.unsqueeze(0).to(device)

cam, logits = gradcam(x)
pred_idx = logits.argmax(dim=1).item()
true_idx = sample_y

img = denormalize(x, stats[0], stats[1])[0].permute(1, 2, 0).detach().cpu().numpy()
heat = cam[0, 0].detach().cpu().numpy()
heat_rgb = plt.cm.jet(heat)[..., :3]
overlay = 0.6 * img + 0.4 * heat_rgb
overlay = overlay.clip(0, 1)

class_names = test_dataset.classes

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title(f"Original\nTrue: {class_names[true_idx]}")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(heat, cmap='jet')
plt.title('GradCAM Heatmap')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title(f"Overlay\nPred: {class_names[pred_idx]}")
plt.axis('off')
plt.tight_layout()
plt.show()

gradcam.close()